In [ ]:


import cv2
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
import pywt

def to_gray(img):
    if img is None:
        return None
    arr = np.array(img)
    if arr.ndim == 2:
        g = arr
    else:
        if arr.shape[-1] == 4:
            arr = arr[..., :3]
        g = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    return g.astype(np.uint8)

def ensure_odd(k):
    k = int(k)
    if k < 1:
        k = 1
    if k % 2 == 0:
        k += 1
    return k

def clip_u8(x):
    return np.clip(x, 0, 255).astype(np.uint8)

def plot_curve(x, y, title, xlabel="Input", ylabel="Output"):
    fig, ax = plt.subplots(figsize=(5.4, 3.2))
    ax.plot(x, y, color="#e91e63", linewidth=2)
    ax.set_title(title, color="white")
    ax.set_xlabel(xlabel, color="white")
    ax.set_ylabel(ylabel, color="white")
    ax.grid(True, alpha=0.2, linestyle="--")
    ax.tick_params(colors="white")
    fig.patch.set_facecolor("#1a1a1a")
    ax.set_facecolor("#2d2d2d")
    fig.tight_layout()
    return fig

def plot_hist(img, title="Histogram"):
    fig, ax = plt.subplots(figsize=(5.4, 3.2))
    ax.hist(img.flatten(), bins=256, range=[0, 255], color="#ec407a", alpha=0.7)
    ax.set_title(title, color="white")
    ax.set_xlabel("Intensity", color="white")
    ax.set_ylabel("Count", color="white")
    ax.tick_params(colors="white")
    fig.patch.set_facecolor("#1a1a1a")
    ax.set_facecolor("#2d2d2d")
    fig.tight_layout()
    return fig

def plot_kernel(kernel, title="Kernel / Mask"):
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    im = ax.imshow(kernel, cmap="magma")
    ax.set_title(title, color="white")
    ax.axis("off")
    fig.patch.set_facecolor("#1a1a1a")
    plt.colorbar(im, ax=ax)
    fig.tight_layout()
    return fig

def simple_info_plot(text):
    fig, ax = plt.subplots(figsize=(5.4, 3.2))
    ax.text(0.5, 0.5, text, ha="center", va="center", color="white", fontsize=12)
    ax.axis("off")
    fig.patch.set_facecolor("#1a1a1a")
    return fig

# Point Operations
def negative(img):
    g = to_gray(img)
    out = cv2.bitwise_not(g)
    x = np.arange(256)
    y = 255 - x
    return out, plot_curve(x, y, "Negative Transform")

def threshold(img, t, mode="binary"):
    g = to_gray(img)
    t = int(t)
    thresh_type = cv2.THRESH_BINARY if mode == "binary" else cv2.THRESH_TOZERO
    _, out = cv2.threshold(g, t, 255, thresh_type)
    x = np.arange(256)
    y = np.where(x < t, 0, 255) if mode == "binary" else np.where(x < t, 0, x)
    return out, plot_curve(x, y, f"Threshold ({mode}, t={t})")

def log_transform(img, c):
    g = to_gray(img).astype(np.float32)
    out = float(c) * cv2.log(1.0 + g)
    cv2.normalize(out, out, 0, 255, cv2.NORM_MINMAX)
    x = np.arange(256, dtype=np.float32)
    y = float(c) * np.log1p(x)
    y = (y / (y.max() + 1e-6)) * 255
    return out.astype(np.uint8), plot_curve(np.arange(256), y, f"Log Transform (c={c:g})")

def gamma_transform(img, gamma):
    g = to_gray(img)
    gamma = float(gamma)
    x = np.arange(256, dtype=np.float32) / 255.0
    y = np.clip((x ** gamma) * 255.0, 0, 255)
    table = y.astype(np.uint8)
    out = cv2.LUT(g, table)
    return out, plot_curve(np.arange(256), y, f"Gamma Transform (γ={gamma:g})")

def contrast_stretch(img, r1, r2):
    g = to_gray(img).astype(np.float32)
    r1, r2 = float(r1), float(r2)
    out = (g - r1) * (255.0 / max(r2 - r1, 1e-6))
    out = np.clip(out, 0, 255).astype(np.uint8)
    x = np.arange(256, dtype=np.float32)
    y = np.clip((x - r1) * 255 / max(r2 - r1, 1e-6), 0, 255)
    return out, plot_curve(x, y, f"Contrast Stretch (r1={r1:g}, r2={r2:g})")

def gray_slice(img, a, b, keep_outside=False):
    g = to_gray(img)
    a, b = int(a), int(b)
    mask = cv2.inRange(g, a, b)
    if keep_outside:
        out = g.copy()
        out[mask > 0] = 255
    else:
        out = np.zeros_like(g)
        out[mask > 0] = 255
    x = np.arange(256)
    y = x.copy() if keep_outside else np.zeros_like(x)
    y[(x >= a) & (x <= b)] = 255
    return out, plot_curve(x, y, f"Gray Level Slicing [{a},{b}]")

# Histogram Ops
def hist_equalization(img):
    g = to_gray(img)
    out = cv2.equalizeHist(g)
    return out, plot_hist(out, "Histogram After Equalization")

def clahe_equalization(img, clip, tile):
    g = to_gray(img)
    clip = float(clip)
    tile = int(tile)
    tile = max(2, tile)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    out = clahe.apply(g)
    return out, plot_hist(out, f"CLAHE (clip={clip:g}, tile={tile}x{tile})")

def hist_matching(img, aux):
    g = to_gray(img)
    t = to_gray(aux)
    if t is None:
        return g, simple_info_plot("Upload an Aux Image for histogram matching")
    src_values, src_counts = np.unique(g, return_counts=True)
    tmpl_values, tmpl_counts = np.unique(t, return_counts=True)
    src_cdf = np.cumsum(src_counts).astype(np.float64) / g.size
    tmpl_cdf = np.cumsum(tmpl_counts).astype(np.float64) / t.size
    interp_tmpl_values = np.interp(src_cdf, tmpl_cdf, tmpl_values)
    lut = np.zeros(256, dtype=np.uint8)
    lut[src_values] = interp_tmpl_values.astype(np.uint8)
    out = cv2.LUT(g, lut)
    return out, plot_hist(out, "Histogram After Matching")

# Spatial Filters
def mean_filter(img, k):
    g = to_gray(img)
    k = ensure_odd(k)
    out = cv2.blur(g, (k, k))
    kernel = np.ones((k, k), dtype=np.float32) / (k * k)
    return out, plot_kernel(kernel, f"Mean Kernel ({k}x{k})")

def weighted_avg_filter(img):
    g = to_gray(img)
    kernel = np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=np.float32) / 16.0
    out = cv2.filter2D(g, -1, kernel)
    return out, plot_kernel(kernel, "Weighted Average Kernel (3x3)")

def gaussian_filter(img, k, sigma):
    g = to_gray(img)
    k = ensure_odd(k)
    out = cv2.GaussianBlur(g, (k, k), float(sigma))
    return out, plot_hist(out, f"Gaussian Blur (k={k}, σ={sigma:g})")

def median_filter(img, k):
    g = to_gray(img)
    k = ensure_odd(k)
    out = cv2.medianBlur(g, k)
    return out, simple_info_plot("Median Filter Applied\n(Non-linear)")

def adaptive_median_sp(img, max_k):
    g = to_gray(img)
    max_k = ensure_odd(max_k)
    out = g.copy()
    noisy = (out == 0) | (out == 255)
    for k in range(3, max_k + 1, 2):
        if not noisy.any():
            break
        med = cv2.medianBlur(out, k)
        out[noisy] = med[noisy]
        noisy = (out == 0) | (out == 255)
    return out, simple_info_plot(f"Adaptive Median (Salt&Pepper)\nMax window = {max_k}x{max_k}")

def min_filter(img, k):
    g = to_gray(img)
    k = ensure_odd(k)
    kernel = np.ones((k, k), np.uint8)
    out = cv2.erode(g, kernel)
    return out, plot_kernel(kernel, f"Min Filter (Erosion) {k}x{k}")

def max_filter(img, k):
    g = to_gray(img)
    k = ensure_odd(k)
    kernel = np.ones((k, k), np.uint8)
    out = cv2.dilate(g, kernel)
    return out, plot_kernel(kernel, f"Max Filter (Dilation) {k}x{k}")

def laplacian_sharpen(img, alpha):
    g = to_gray(img).astype(np.float32)
    lap = cv2.Laplacian(g, cv2.CV_32F, ksize=3)
    out = cv2.subtract(g, float(alpha) * lap)
    out = np.clip(out, 0, 255).astype(np.uint8)
    return out, plot_hist(out, f"Laplacian Sharpen (α={alpha:g})")

def unsharp_mask(img, k, sigma, amount):
    g = to_gray(img).astype(np.float32)
    k = ensure_odd(k)
    sigma = float(sigma)
    amount = float(amount)
    blur = cv2.GaussianBlur(g, (k, k), sigma)
    mask = g - blur
    out = g + amount * mask
    return clip_u8(out), simple_info_plot(f"Unsharp Mask\nk={k}, σ={sigma:g}, amount={amount:g}")

def high_boost(img, k, sigma, A):
    g = to_gray(img).astype(np.float32)
    k = ensure_odd(k)
    sigma = float(sigma)
    A = float(A)
    blur = cv2.GaussianBlur(g, (k, k), sigma)
    out = A * g - blur
    return clip_u8(out), simple_info_plot(f"High-Boost Sharpen\nk={k}, σ={sigma:g}, A={A:g}")

def edge_detect(img, method, edge_low, edge_high):
    g = to_gray(img).astype(np.float32)

    method = str(method)
    if method == "Sobel":
        gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
        mag = cv2.magnitude(gx, gy)
        out = clip_u8(cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX))
        kx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
        ky = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)
        fig = plot_kernel(kx, "Sobel Gx (3x3)")
        return out, fig

    if method == "Prewitt":
        kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
        ky = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32)
        gx = cv2.filter2D(g, -1, kx)
        gy = cv2.filter2D(g, -1, ky)
        mag = cv2.magnitude(gx, gy)
        out = clip_u8(cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX))
        return out, plot_kernel(kx, "Prewitt Gx (3x3)")

    if method == "Roberts":
        kx = np.array([[1,0],[0,-1]], dtype=np.float32)
        ky = np.array([[0,1],[-1,0]], dtype=np.float32)
        gx = cv2.filter2D(g, -1, kx)
        gy = cv2.filter2D(g, -1, ky)
        mag = cv2.magnitude(gx, gy)
        out = clip_u8(cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX))
        return out, plot_kernel(kx, "Roberts Gx (2x2)")

    u8 = clip_u8(g)
    low = int(edge_low)
    high = int(edge_high)
    out = cv2.Canny(u8, low, high)
    return out, simple_info_plot(f"Canny Edges\nlow={low}, high={high}")

# Frequency Domain
def _fft_mask(h, w, mode, cutoff1, cutoff2, ftype, order, notch_dx, notch_dy, notch_r):
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    D = np.sqrt((Y - cy) ** 2 + (X - cx) ** 2).astype(np.float32)

    cutoff1 = float(cutoff1)
    cutoff2 = float(cutoff2)
    order = max(1, int(order))

    def ideal_lp(D0):
        return (D <= D0).astype(np.float32)

    def gaussian_lp(D0):
        D0 = max(D0, 1e-6)
        return np.exp(-(D * D) / (2.0 * (D0 * D0))).astype(np.float32)

    def butter_lp(D0, n):
        D0 = max(D0, 1e-6)
        return (1.0 / (1.0 + (D / D0) ** (2 * n))).astype(np.float32)

    if ftype == "ideal":
        Hlp1 = ideal_lp(cutoff1)
        Hlp2 = ideal_lp(cutoff2)
    elif ftype == "gaussian":
        Hlp1 = gaussian_lp(cutoff1)
        Hlp2 = gaussian_lp(cutoff2)
    else:
        Hlp1 = butter_lp(cutoff1, order)
        Hlp2 = butter_lp(cutoff2, order)

    if mode == "lowpass":
        H = Hlp1
    elif mode == "highpass":
        H = 1.0 - Hlp1
    elif mode == "bandpass":
        lo = min(cutoff1, cutoff2)
        hi = max(cutoff1, cutoff2)
        if ftype == "ideal":
            H = ((D >= lo) & (D <= hi)).astype(np.float32)
        else:
            if ftype == "gaussian":
                Hhi = gaussian_lp(hi)
                Hlo = gaussian_lp(lo)
            else:
                Hhi = butter_lp(hi, order)
                Hlo = butter_lp(lo, order)
            H = Hhi * (1.0 - Hlo)
    elif mode == "bandreject":
        lo = min(cutoff1, cutoff2)
        hi = max(cutoff1, cutoff2)
        if ftype == "ideal":
            H = 1.0 - (((D >= lo) & (D <= hi)).astype(np.float32))
        else:
            if ftype == "gaussian":
                Hhi = gaussian_lp(hi)
                Hlo = gaussian_lp(lo)
            else:
                Hhi = butter_lp(hi, order)
                Hlo = butter_lp(lo, order)
            Hbp = Hhi * (1.0 - Hlo)
            H = 1.0 - Hbp
    else:
        H = np.ones((h, w), dtype=np.float32)
        r = max(1.0, float(notch_r))
        for sx, sy in [(cx + notch_dx, cy + notch_dy), (cx - notch_dx, cy - notch_dy)]:
            d2 = np.sqrt((Y - sy) ** 2 + (X - sx) ** 2)
            H[d2 <= r] = 0.0

    return H

def fft_filter_advanced(img, fft_mode, cutoff, cutoff2, fft_type, butter_n, notch_dx, notch_dy, notch_r):
    g = to_gray(img).astype(np.float32)
    h, w = g.shape

    dft = cv2.dft(g, flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)

    H = _fft_mask(h, w, fft_mode, cutoff, cutoff2, fft_type, butter_n, notch_dx, notch_dy, notch_r)

    dft_shift[:, :, 0] *= H
    dft_shift[:, :, 1] *= H

    f_ishift = np.fft.ifftshift(dft_shift)
    img_back = cv2.idft(f_ishift)
    out = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])
    cv2.normalize(out, out, 0, 255, cv2.NORM_MINMAX)
    out = out.astype(np.uint8)

    fig, axs = plt.subplots(1, 2, figsize=(10, 3.3))
    mag_spectrum = 20 * np.log(cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1]) + 1)
    axs[0].imshow(mag_spectrum, cmap="gray")
    axs[0].set_title("Spectrum (log)", color="white")
    axs[0].axis("off")
    axs[1].imshow(H, cmap="gray")
    axs[1].set_title(f"Mask ({fft_mode}, {fft_type})", color="white")
    axs[1].axis("off")
    fig.patch.set_facecolor("#1a1a1a")
    fig.tight_layout()
    return out, fig

# Wavelets
def wavelet_decompose(img, wave):
    g = to_gray(img).astype(np.float32)
    cA, (cH, cV, cD) = pywt.dwt2(g, wave)

    def norm(x):
        return cv2.normalize(x, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    LL, LH, HL, HH = norm(cA), norm(cH), norm(cV), norm(cD)

    fig, axs = plt.subplots(1, 4, figsize=(10.5, 3.0))
    for ax, im, title in zip(axs, [LL, LH, HL, HH], ["LL", "LH", "HL", "HH"]):
        ax.imshow(im, cmap="gray")
        ax.set_title(title, color="white")
        ax.axis("off")
    fig.patch.set_facecolor("#1a1a1a")
    fig.suptitle(f"DWT2 Subbands ({wave})", color="white", y=1.03)
    fig.tight_layout()

    rec = pywt.idwt2((cA, (cH, cV, cD)), wave)
    rec = np.clip(rec, 0, 255).astype(np.uint8)
    return rec, fig

def wavelet_denoise(img, wave, level, thresh):
    g = to_gray(img).astype(np.float32)
    coeffs = pywt.wavedec2(g, wavelet=wave, level=int(level))
    new_coeffs = [coeffs[0]]
    for detail_coeffs in coeffs[1:]:
        new_coeffs.append(tuple(pywt.threshold(c, float(thresh), mode="soft") for c in detail_coeffs))
    rec = pywt.waverec2(new_coeffs, wavelet=wave)
    rec = np.clip(rec, 0, 255).astype(np.uint8)
    return rec, plot_hist(rec, f"Wavelet Denoise (lvl={level}, thr={thresh:g})")

def wavelet_packet_decompose(img, wave, level):
    g = to_gray(img).astype(np.float32)
    level = int(level)
    level = max(1, min(level, 3))

    wp = pywt.WaveletPacket2D(data=g, wavelet=wave, mode="symmetric", maxlevel=level)
    nodes = wp.get_level(level, order="natural")

    grid = 2 ** level

    fig, axs = plt.subplots(grid, grid, figsize=(10, 10))
    axs = np.array(axs).reshape(grid, grid)

    for idx, node in enumerate(nodes):
        r = idx // grid
        c = idx % grid
        band = node.data
        band_u8 = cv2.normalize(band, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        axs[r, c].imshow(band_u8, cmap="gray")
        axs[r, c].set_title(node.path, color="white", fontsize=9)
        axs[r, c].axis("off")

    fig.patch.set_facecolor("#1a1a1a")
    fig.suptitle(f"Wavelet Packet 2D (level={level}, wave={wave})", color="white")
    fig.tight_layout()

    rec = wp.reconstruct(update=False)
    rec = np.clip(rec, 0, 255).astype(np.uint8)
    return rec, fig

# Noise Models
def add_noise_models(img, noise_type, amount, n_p1, n_p2):
    g = to_gray(img).astype(np.float32)
    amount = float(amount)
    n_p1 = float(n_p1)
    n_p2 = float(n_p2)

    if noise_type == "gaussian":
        noise = np.zeros(g.shape, np.float32)
        cv2.randn(noise, 0, 255.0 * amount)
        out = g + noise
        out = clip_u8(out)
        return out, plot_hist(out, "Noise Added: Gaussian")

    if noise_type == "salt_pepper":
        out = g.copy()
        rnd = np.random.rand(*g.shape)
        out[rnd < amount / 2.0] = 0
        out[(rnd >= amount / 2.0) & (rnd < amount)] = 255
        out = out.astype(np.uint8)
        return out, plot_hist(out, "Noise Added: Salt & Pepper")

    if noise_type == "uniform":
        a = 255.0 * amount
        noise = np.random.uniform(-a, a, size=g.shape).astype(np.float32)
        out = clip_u8(g + noise)
        return out, plot_hist(out, "Noise Added: Uniform")

    if noise_type == "rayleigh":
        b = n_p1 if n_p1 > 0 else (255.0 * amount + 1.0)
        noise = np.random.rayleigh(scale=b, size=g.shape).astype(np.float32)
        out = clip_u8(g + noise)
        return out, plot_hist(out, f"Noise Added: Rayleigh (scale={b:g})")

    if noise_type == "exponential":
        lam = max(1e-6, n_p1)
        noise = np.random.exponential(scale=1.0 / lam, size=g.shape).astype(np.float32)
        noise = noise * (255.0 * amount)
        out = clip_u8(g + noise)
        return out, plot_hist(out, f"Noise Added: Exponential (λ={lam:g})")

    k = max(0.1, n_p1)
    theta = max(1e-6, n_p2)
    noise = np.random.gamma(shape=k, scale=theta, size=g.shape).astype(np.float32)
    noise = noise * (255.0 * amount)
    out = clip_u8(g + noise)
    return out, plot_hist(out, f"Noise Added: Gamma (k={k:g}, θ={theta:g})")

def geometric_mean_filter(img, k):
    g = to_gray(img).astype(np.float32) + 1e-6
    k = ensure_odd(k)
    log_img = cv2.log(g)
    mean_log = cv2.blur(log_img, (k, k))
    out = np.exp(mean_log)
    out = clip_u8(out)
    return out, simple_info_plot(f"Geometric Mean {k}x{k}\nexp(mean(log(I)))")

def harmonic_mean_filter(img, k):
    g = to_gray(img).astype(np.float32) + 1e-6
    k = ensure_odd(k)
    inv = 1.0 / g
    mean_inv = cv2.blur(inv, (k, k))
    out = 1.0 / (mean_inv + 1e-6)
    out = clip_u8(out)
    return out, simple_info_plot(f"Harmonic Mean {k}x{k}\n1 / mean(1/I)")

def contraharmonic_mean_filter(img, k, q):
    g = to_gray(img).astype(np.float32) + 1e-6
    k = ensure_odd(k)
    q = float(q)
    num = cv2.blur(np.power(g, q + 1.0), (k, k))
    den = cv2.blur(np.power(g, q), (k, k))
    out = num / (den + 1e-6)
    out = clip_u8(out)
    return out, simple_info_plot(f"Contraharmonic Mean\nQ={q:g}, k={k}")

# Morphology
def morph_op(img, op_name, shape, k):
    g = to_gray(img)
    k = ensure_odd(k)
    s_elem = cv2.MORPH_RECT if shape == "Square" else (cv2.MORPH_CROSS if shape == "Cross" else cv2.MORPH_ELLIPSE)
    se = cv2.getStructuringElement(s_elem, (k, k))

    ops = {
        "Erosion": cv2.MORPH_ERODE, "Dilation": cv2.MORPH_DILATE,
        "Opening": cv2.MORPH_OPEN, "Closing": cv2.MORPH_CLOSE,
        "Morph Gradient": cv2.MORPH_GRADIENT, "Top Hat": cv2.MORPH_TOPHAT, "Black Hat": cv2.MORPH_BLACKHAT
    }

    if op_name in ["Erosion", "Dilation"]:
        out = cv2.erode(g, se) if op_name == "Erosion" else cv2.dilate(g, se)
    else:
        out = cv2.morphologyEx(g, ops[op_name], se)

    return out, plot_kernel(se, f"{op_name} SE ({shape}, {k}x{k})")

def zhang_suen_thinning(bin_img01, max_iter=100):
    img = np.pad(bin_img01.astype(np.uint8), 1, mode="constant")
    it = 0
    changed = True

    while changed and it < max_iter:
        changed = False
        for step in [0, 1]:
            P1 = img[1:-1, 1:-1]
            P2 = img[0:-2, 1:-1]
            P3 = img[0:-2, 2:]
            P4 = img[1:-1, 2:]
            P5 = img[2:, 2:]
            P6 = img[2:, 1:-1]
            P7 = img[2:, 0:-2]
            P8 = img[1:-1, 0:-2]
            P9 = img[0:-2, 0:-2]

            N = P2 + P3 + P4 + P5 + P6 + P7 + P8 + P9

            T = ((P2 == 0) & (P3 == 1)).astype(np.uint8) + \
                ((P3 == 0) & (P4 == 1)).astype(np.uint8) + \
                ((P4 == 0) & (P5 == 1)).astype(np.uint8) + \
                ((P5 == 0) & (P6 == 1)).astype(np.uint8) + \
                ((P6 == 0) & (P7 == 1)).astype(np.uint8) + \
                ((P7 == 0) & (P8 == 1)).astype(np.uint8) + \
                ((P8 == 0) & (P9 == 1)).astype(np.uint8) + \
                ((P9 == 0) & (P2 == 1)).astype(np.uint8)

            if step == 0:
                m = (P1 == 1) & (N >= 2) & (N <= 6) & (T == 1) & ((P2 * P4 * P6) == 0) & ((P4 * P6 * P8) == 0)
            else:
                m = (P1 == 1) & (N >= 2) & (N <= 6) & (T == 1) & ((P2 * P4 * P8) == 0) & ((P2 * P6 * P8) == 0)

            if np.any(m):
                changed = True
                img[1:-1, 1:-1][m] = 0

        it += 1

    return img[1:-1, 1:-1]

def skeletonize(img, t, max_iter):
    g = to_gray(img)
    t = int(t)
    max_iter = int(max_iter)
    _, b = cv2.threshold(g, t, 1, cv2.THRESH_BINARY)
    thin = zhang_suen_thinning(b, max_iter=max_iter)
    out = (thin * 255).astype(np.uint8)
    return out, simple_info_plot(f"Skeletonize (Zhang-Suen)\nthreshold={t}, max_iter={max_iter}")

def binary_set_op(img, aux, op, t):
    g1 = to_gray(img)
    g2 = to_gray(aux)
    if g2 is None:
        return g1, simple_info_plot("Upload Aux Image for binary set operation")
    t = int(t)
    _, b1 = cv2.threshold(g1, t, 255, cv2.THRESH_BINARY)
    _, b2 = cv2.threshold(g2, t, 255, cv2.THRESH_BINARY)

    if op == "Complement":
        out = cv2.bitwise_not(b1)
    elif op == "Union":
        out = cv2.bitwise_or(b1, b2)
    elif op == "Intersection":
        out = cv2.bitwise_and(b1, b2)
    else:
        out = cv2.bitwise_and(b1, cv2.bitwise_not(b2))

    return out, simple_info_plot(f"Binary Set Op: {op}\n(threshold={t})")

# UI
FILTER_GROUPS = {
    "Point Operations": [
        "Negative", "Threshold", "Log Transform", "Gamma Transform",
        "Contrast Stretch", "Gray Slicing"
    ],
    "Histogram": [
        "Histogram Equalization", "CLAHE (Local Equalization)", "Histogram Matching"
    ],
    "Spatial Filters": [
        "Mean Filter", "Weighted Average", "Gaussian Filter", "Median Filter",
        "Adaptive Median (Salt&Pepper)",
        "Min Filter", "Max Filter",
        "Laplacian Sharpen", "Unsharp Mask", "High-Boost Sharpen",
        "Edge Detection (Gradient)"
    ],
    "Frequency Domain": [
        "Frequency Filtering (FFT Advanced)"
    ],
    "Wavelets": [
        "Wavelet Decompose", "Wavelet Denoise", "Wavelet Packet Decompose"
    ],
    "Noise & Restoration": [
        "Add Noise (Models)", "Geometric Mean", "Harmonic Mean", "Contraharmonic Mean"
    ],
    "Morphological": [
        "Erosion", "Dilation", "Opening", "Closing",
        "Morph Gradient", "Top Hat", "Black Hat",
        "Skeletonize (Thinning)"
    ],
    "Binary Set Ops": [
        "Binary Complement", "Binary Union", "Binary Intersection", "Binary Difference"
    ]
}

FILTER_SPECS = {
    # Point
    "Negative": dict(fn=lambda img, p: negative(img), params=[]),
    "Threshold": dict(fn=lambda img, p: threshold(img, p["t"], p["thr_mode"]), params=["t", "thr_mode"]),
    "Log Transform": dict(fn=lambda img, p: log_transform(img, p["c"]), params=["c"]),
    "Gamma Transform": dict(fn=lambda img, p: gamma_transform(img, p["gamma"]), params=["gamma"]),
    "Contrast Stretch": dict(fn=lambda img, p: contrast_stretch(img, p["r1"], p["r2"]), params=["r1", "r2"]),
    "Gray Slicing": dict(fn=lambda img, p: gray_slice(img, p["a"], p["b"], p["keep"]), params=["a", "b", "keep"]),

    # Histogram
    "Histogram Equalization": dict(fn=lambda img, p: hist_equalization(img), params=[]),
    "CLAHE (Local Equalization)": dict(fn=lambda img, p: clahe_equalization(img, p["clahe_clip"], p["clahe_tile"]), params=["clahe_clip", "clahe_tile"]),
    "Histogram Matching": dict(fn=lambda img, p: hist_matching(img, p["aux"]), params=["aux"]),

    # Spatial
    "Mean Filter": dict(fn=lambda img, p: mean_filter(img, p["k"]), params=["k"]),
    "Weighted Average": dict(fn=lambda img, p: weighted_avg_filter(img), params=[]),
    "Gaussian Filter": dict(fn=lambda img, p: gaussian_filter(img, p["k"], p["sigma"]), params=["k", "sigma"]),
    "Median Filter": dict(fn=lambda img, p: median_filter(img, p["k"]), params=["k"]),
    "Adaptive Median (Salt&Pepper)": dict(fn=lambda img, p: adaptive_median_sp(img, p["k"]), params=["k"]),
    "Min Filter": dict(fn=lambda img, p: min_filter(img, p["k"]), params=["k"]),
    "Max Filter": dict(fn=lambda img, p: max_filter(img, p["k"]), params=["k"]),
    "Laplacian Sharpen": dict(fn=lambda img, p: laplacian_sharpen(img, p["alpha"]), params=["alpha"]),
    "Unsharp Mask": dict(fn=lambda img, p: unsharp_mask(img, p["k"], p["sigma"], p["us_amount"]), params=["k", "sigma", "us_amount"]),
    "High-Boost Sharpen": dict(fn=lambda img, p: high_boost(img, p["k"], p["sigma"], p["hb_A"]), params=["k", "sigma", "hb_A"]),
    "Edge Detection (Gradient)": dict(fn=lambda img, p: edge_detect(img, p["edge_method"], p["edge_low"], p["edge_high"]),
                                      params=["edge_method", "edge_low", "edge_high"]),

    # FFT advanced
    "Frequency Filtering (FFT Advanced)": dict(
        fn=lambda img, p: fft_filter_advanced(
            img, p["fft_mode"], p["cutoff"], p["cutoff2"], p["fft_type"], p["butter_n"],
            p["notch_dx"], p["notch_dy"], p["notch_r"]
        ),
        params=["fft_mode", "cutoff", "cutoff2", "fft_type", "butter_n", "notch_dx", "notch_dy", "notch_r"]
    ),

    # Wavelets
    "Wavelet Decompose": dict(fn=lambda img, p: wavelet_decompose(img, p["wave"]), params=["wave"]),
    "Wavelet Denoise": dict(fn=lambda img, p: wavelet_denoise(img, p["wave"], p["level"], p["thr"]), params=["wave", "level", "thr"]),
    "Wavelet Packet Decompose": dict(fn=lambda img, p: wavelet_packet_decompose(img, p["wave"], p["wp_level"]), params=["wave", "wp_level"]),

    # Noise models
    "Add Noise (Models)": dict(fn=lambda img, p: add_noise_models(img, p["noise_type"], p["amount"], p["n_p1"], p["n_p2"]),
                               params=["noise_type", "amount", "n_p1", "n_p2"]),
    "Geometric Mean": dict(fn=lambda img, p: geometric_mean_filter(img, p["k"]), params=["k"]),
    "Harmonic Mean": dict(fn=lambda img, p: harmonic_mean_filter(img, p["k"]), params=["k"]),
    "Contraharmonic Mean": dict(fn=lambda img, p: contraharmonic_mean_filter(img, p["k"], p["q"]), params=["k", "q"]),

    # Morphology
    "Erosion": dict(fn=lambda img, p: morph_op(img, "Erosion", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Dilation": dict(fn=lambda img, p: morph_op(img, "Dilation", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Opening": dict(fn=lambda img, p: morph_op(img, "Opening", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Closing": dict(fn=lambda img, p: morph_op(img, "Closing", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Morph Gradient": dict(fn=lambda img, p: morph_op(img, "Morph Gradient", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Top Hat": dict(fn=lambda img, p: morph_op(img, "Top Hat", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Black Hat": dict(fn=lambda img, p: morph_op(img, "Black Hat", p["se_shape"], p["k"]), params=["se_shape", "k"]),
    "Skeletonize (Thinning)": dict(fn=lambda img, p: skeletonize(img, p["t"], p["sk_max_iter"]), params=["t", "sk_max_iter"]),

    # Binary set ops (uses aux image)
    "Binary Complement": dict(fn=lambda img, p: binary_set_op(img, p["aux"], "Complement", p["t"]), params=["aux", "t"]),
    "Binary Union": dict(fn=lambda img, p: binary_set_op(img, p["aux"], "Union", p["t"]), params=["aux", "t"]),
    "Binary Intersection": dict(fn=lambda img, p: binary_set_op(img, p["aux"], "Intersection", p["t"]), params=["aux", "t"]),
    "Binary Difference": dict(fn=lambda img, p: binary_set_op(img, p["aux"], "Difference", p["t"]), params=["aux", "t"]),
}

PARAM_KEYS = [
    "t","thr_mode","c","gamma","r1","r2","a","b","keep",
    "aux","k","sigma","alpha",
    "clahe_clip","clahe_tile",
    "us_amount","hb_A",
    "edge_method","edge_low","edge_high",
    "se_shape",
    "fft_mode","fft_type","cutoff","cutoff2","butter_n","notch_dx","notch_dy","notch_r",
    "wave","level","thr","wp_level",
    "noise_type","amount","n_p1","n_p2","q",
    "sk_max_iter"
]

def toggle_params(selected_name):
    if not selected_name:
        return [gr.update(visible=False)] * len(PARAM_KEYS)
    needed = set(FILTER_SPECS[selected_name]["params"])
    def v(key):
        return gr.update(visible=(key in needed))
    return [v(k) for k in PARAM_KEYS]

def run_selected(img, selected_name, *vals):
    if img is None or not selected_name:
        return None, None
    p = dict(zip(PARAM_KEYS, vals))
    return FILTER_SPECS[selected_name]["fn"](img, p)

def run_all(img, *vals, progress=gr.Progress()):
    if img is None:
        return []
    p = dict(zip(PARAM_KEYS, vals))
    results = []
    all_filters = [name for group in FILTER_GROUPS.values() for name in group]
    for i, name in enumerate(all_filters):
        progress((i / max(1, len(all_filters))), desc=f"Processing: {name}")
        if "aux" in FILTER_SPECS[name]["params"] and p.get("aux", None) is None:
            continue
        try:
            out, _ = FILTER_SPECS[name]["fn"](img, p)
            results.append((out, name))
        except:
            continue
    return results

CSS = """
#center_header{ text-align:center; margin-bottom: 20px; color: #ec407a; }
body{ background-color: #0b0f1a !important; }
.card{ border: 1px solid rgba(236, 64, 122, 0.2); border-radius: 12px; background: rgba(255,255,255,0.03); padding: 15px; }
#filters_sidebar{ height: 80vh; overflow-y: auto; }
footer {display: none !important;}
"""

with gr.Blocks(css=CSS, theme=gr.themes.Default(primary_hue="pink", secondary_hue="pink")) as demo:
    gr.Markdown(
        ""
        "🧪 Imaging Filters Lab"
        "Upload an image • Choose a filter • Tune parameters • Apply"
        ""
    )

    with gr.Row():
        # LEFT
        with gr.Column(scale=3, elem_classes=["card"]):
            gr.Markdown("### 🛠️ Select Filter")
            selected_name = gr.Textbox(visible=False, value=None)

            pickers = []
            for group_name, filters in FILTER_GROUPS.items():
                with gr.Accordion(group_name, open=(group_name == "Point Operations")):
                    p = gr.Radio(filters, label="", value=None)
                    pickers.append(p)

            with gr.Row():
                clear_btn = gr.Button("🧹 Reset", variant="secondary")
                apply_all_btn = gr.Button("🚀 Batch Process", variant="primary")

        # CENTER
        with gr.Column(scale=6):
            with gr.Row():
                img_in = gr.Image(type="numpy", label="Input Image", height=400)
                out_img = gr.Image(label="Processed Result", height=400, show_download_button=True)

            with gr.Tabs():
                with gr.TabItem("📊 Analysis"):
                    plot_out = gr.Plot(label="Visualization")
                with gr.TabItem("🖼️ Batch Gallery"):
                    gallery = gr.Gallery(label="All Results", columns=4, height=500, preview=True)

        # RIGHT
        with gr.Column(scale=3, elem_classes=["card"]):
            gr.Markdown("### ⚙️ Parameters")

            # Common
            t = gr.Slider(0, 255, 127, label="Threshold (for Binary / Skeleton)", visible=False)
            thr_mode = gr.Radio(["binary", "tozero"], value="binary", label="Thresh Mode", visible=False)
            c = gr.Slider(1, 100, 25, label="Log Constant (C)", visible=False)
            gamma = gr.Slider(0.1, 5.0, 1.0, label="Gamma Value", visible=False)
            r1 = gr.Slider(0, 255, 50, label="Min Intensity (r1)", visible=False)
            r2 = gr.Slider(0, 255, 200, label="Max Intensity (r2)", visible=False)
            a = gr.Slider(0, 255, 100, label="Slice Start (A)", visible=False)
            b = gr.Slider(0, 255, 200, label="Slice End (B)", visible=False)
            keep = gr.Checkbox(False, label="Keep Background", visible=False)

            aux = gr.Image(type="numpy", label="Aux Image (Template / Binary ops)", visible=False)

            k = gr.Slider(3, 51, 5, step=2, label="Kernel Size (K)", visible=False)
            sigma = gr.Slider(0.1, 10, 1.5, label="Gaussian Sigma", visible=False)
            alpha = gr.Slider(0.1, 5.0, 1.0, label="Laplacian Alpha", visible=False)

            # CLAHE
            clahe_clip = gr.Slider(0.5, 10.0, 2.0, label="CLAHE ClipLimit", visible=False)
            clahe_tile = gr.Slider(2, 32, 8, step=1, label="CLAHE Tile Size", visible=False)

            # Unsharp / High-boost
            us_amount = gr.Slider(0.0, 5.0, 1.0, label="Unsharp Amount", visible=False)
            hb_A = gr.Slider(1.0, 5.0, 1.5, label="High-Boost A", visible=False)

            # Edges
            edge_method = gr.Dropdown(["Sobel", "Prewitt", "Roberts", "Canny"], value="Sobel", label="Edge Method", visible=False)
            edge_low = gr.Slider(0, 255, 80, label="Canny Low", visible=False)
            edge_high = gr.Slider(0, 255, 160, label="Canny High", visible=False)

            # Morphology
            se_shape = gr.Dropdown(["Square", "Cross", "Disk"], value="Square", label="SE Shape", visible=False)

            # FFT Advanced
            fft_mode = gr.Dropdown(["lowpass", "highpass", "bandpass", "bandreject", "notch"], value="lowpass", label="FFT Mode", visible=False)
            fft_type = gr.Dropdown(["ideal", "gaussian", "butterworth"], value="ideal", label="FFT Filter Type", visible=False)
            cutoff = gr.Slider(5, 300, 50, label="Cutoff 1 (D0)", visible=False)
            cutoff2 = gr.Slider(5, 300, 120, label="Cutoff 2 (for band)", visible=False)
            butter_n = gr.Slider(1, 10, 2, step=1, label="Butterworth Order (n)", visible=False)
            notch_dx = gr.Slider(0, 300, 50, label="Notch DX", visible=False)
            notch_dy = gr.Slider(0, 300, 50, label="Notch DY", visible=False)
            notch_r = gr.Slider(1, 80, 10, label="Notch Radius", visible=False)

            # Wavelets
            wave = gr.Dropdown(["haar", "db1", "db2", "sym2"], value="haar", label="Wavelet Type", visible=False)
            level = gr.Slider(1, 5, 2, step=1, label="Denoise Decomposition Level", visible=False)
            thr = gr.Slider(1, 100, 20, label="Denoise Threshold", visible=False)
            wp_level = gr.Slider(1, 3, 2, step=1, label="Wavelet Packet Level", visible=False)

            # Noise
            noise_type = gr.Radio(
                ["gaussian", "salt_pepper", "uniform", "rayleigh", "exponential", "gamma"],
                value="gaussian",
                label="Noise Type",
                visible=False
            )
            amount = gr.Slider(0.0, 0.5, 0.05, label="Noise Amount (general)", visible=False)
            n_p1 = gr.Slider(0.1, 50.0, 10.0, label="Noise Param 1 (Rayleigh scale / Exp lambda / Gamma k)", visible=False)
            n_p2 = gr.Slider(0.1, 10.0, 1.0, label="Noise Param 2 (Gamma θ)", visible=False)

            q = gr.Slider(-5.0, 5.0, 1.5, label="Order (Q) - Contraharmonic", visible=False)

            # Skeleton
            sk_max_iter = gr.Slider(5, 150, 50, step=1, label="Skeleton Max Iterations", visible=False)

            apply_btn = gr.Button("✨ Apply Filter", variant="primary", size="lg")

    def update_selection(*args):
        current_selected = args[-1]
        new_val = None
        for val in args[:-1]:
            if val is not None:
                new_val = val
                break
        res = [new_val]
        for val in args[:-1]:
            res.append(new_val if val == new_val else None)
        return tuple(res)

    def clear_selection():
        return tuple([None] * (len(pickers) + 1))

    for p in pickers:
        p.change(update_selection, inputs=pickers + [selected_name], outputs=[selected_name] + pickers)

    clear_btn.click(clear_selection, outputs=[selected_name] + pickers)

    selected_name.change(
        toggle_params,
        inputs=[selected_name],
        outputs=[
            t, thr_mode, c, gamma, r1, r2, a, b, keep,
            aux, k, sigma, alpha,
            clahe_clip, clahe_tile,
            us_amount, hb_A,
            edge_method, edge_low, edge_high,
            se_shape,
            fft_mode, fft_type, cutoff, cutoff2, butter_n, notch_dx, notch_dy, notch_r,
            wave, level, thr, wp_level,
            noise_type, amount, n_p1, n_p2, q,
            sk_max_iter
        ]
    )

    apply_btn.click(
        run_selected,
        inputs=[
            img_in, selected_name,
            t, thr_mode, c, gamma, r1, r2, a, b, keep,
            aux, k, sigma, alpha,
            clahe_clip, clahe_tile,
            us_amount, hb_A,
            edge_method, edge_low, edge_high,
            se_shape,
            fft_mode, fft_type, cutoff, cutoff2, butter_n, notch_dx, notch_dy, notch_r,
            wave, level, thr, wp_level,
            noise_type, amount, n_p1, n_p2, q,
            sk_max_iter
        ],
        outputs=[out_img, plot_out]
    )

    apply_all_btn.click(
        run_all,
        inputs=[
            img_in,
            t, thr_mode, c, gamma, r1, r2, a, b, keep,
            aux, k, sigma, alpha,
            clahe_clip, clahe_tile,
            us_amount, hb_A,
            edge_method, edge_low, edge_high,
            se_shape,
            fft_mode, fft_type, cutoff, cutoff2, butter_n, notch_dx, notch_dy, notch_r,
            wave, level, thr, wp_level,
            noise_type, amount, n_p1, n_p2, q,
            sk_max_iter
        ],
        outputs=[gallery]
    )

if __name__ == "__main__":
    demo.launch()

